In [ ]:
# 1. IMPORT LIBRARY
import pandas as pd
import numpy as np
import math

In [ ]:
# 2. LOAD DATA

df = pd.read_csv('data green economy.csv')

# Ambil kolom dokumen (kolom ke-2)
documents_raw = df.iloc[:, 1].dropna().tolist()

# Lowercase agar konsisten
documents_raw = [doc.lower() for doc in documents_raw]

# Tokenisasi
corpus = [doc.split() for doc in documents_raw]

print(f"Jumlah dokumen: {len(corpus)}")
print(f"Contoh dokumen pertama: {documents_raw[0][:100]}...")

Jumlah dokumen: 50
Contoh dokumen pertama: bergerak menuju ekonomi hijau berarti mengganggu jaringan perdagangan global yang sudah mapan dan ni...


In [ ]:
# 3. FUNCTION TF (Normalized)

def term_frequency(word, document):
    """TF dinormalisasi: frekuensi kata dibagi total kata dalam dokumen."""
    count = document.count(word)
    if len(document) == 0:
        return 0
    return count / len(document)

# 4. FUNCTION TF LOG-SCALED 1 + log10(tf) jika tf > 0, else 0

def tf_log_scaled(word, document):
    """Log frequency weighting untuk menghindari dominasi raw count."""
    count = document.count(word)
    if count > 0:
        return 1 + math.log10(count)
    return 0


# Gunakan versi log-scaled sebagai TF utama
tf = tf_log_scaled

print("Fungsi TF berhasil didefinisikan.")
print(f"Contoh: tf('green', corpus[0]) = {tf('green', corpus[0]):.4f}")

Fungsi TF berhasil didefinisikan.
Contoh: tf('green', corpus[0]) = 0.0000


In [ ]:
# 5. FUNCTION IDF (Safe / Smooth) log10((N+1) / (df+1)) + 1

def inverse_document_frequency(word, corpus):
    """IDF dengan Laplace smoothing agar aman dari division by zero."""
    N = len(corpus) + 1                                          # total dokumen + 1
    df = sum(1 for doc in corpus if word in doc) + 1            # dokumen yang mengandung kata + 1
    return math.log10(N / df) + 1


idf = inverse_document_frequency

print("Fungsi IDF berhasil didefinisikan.")
print(f"Contoh: idf('green', corpus) = {idf('green', corpus):.4f}")

Fungsi IDF berhasil didefinisikan.
Contoh: idf('green', corpus) = 1.0948


In [ ]:
# 6. FUNCTION TF-IDF

def tf_idf(word, document, corpus):
    """Bobot akhir: TF x IDF."""
    return tf(word, document) * idf(word, corpus)


print("Fungsi TF-IDF berhasil didefinisikan.")
print(f"Contoh: tf_idf('green', corpus[0], corpus) = {tf_idf('green', corpus[0], corpus):.4f}")

Fungsi TF-IDF berhasil didefinisikan.
Contoh: tf_idf('green', corpus[0], corpus) = 0.0000


In [ ]:
# 7. FUNCTION VSM — DOT PRODUCT Σ (Wt,Q × Wt,D)

def vsm_score(query, document, corpus):
    """Hitung kemiripan query-dokumen menggunakan dot product TF-IDF."""
    score = 0
    for word in query:
        w_query = tf_idf(word, query, corpus)       # bobot kata di query
        w_doc   = tf_idf(word, document, corpus)    # bobot kata di dokumen
        score  += w_query * w_doc
    return score


print("Fungsi VSM berhasil didefinisikan.")

Fungsi VSM berhasil didefinisikan.


In [ ]:
# 8. INPUT QUERY

query_input = input("Masukkan query: ")
query = query_input.lower().split()    # lowercase agar konsisten dengan corpus

print(f"\nQuery: {query}")

Masukkan query: economy

Query: ['economy']


In [ ]:
# 9. HITUNG SKOR SEMUA DOKUMEN

results = []

for i, doc in enumerate(corpus):
    score = vsm_score(query, doc, corpus)
    score = round(score, 5)    # pembulatan untuk grouping
    results.append((i + 1, documents_raw[i], score))

# Sort descending
results = sorted(results, key=lambda x: x[2], reverse=True)

print(f"Skor berhasil dihitung untuk {len(results)} dokumen.")

Skor berhasil dihitung untuk 50 dokumen.


In [ ]:

# 10. GROUPING DAN TAMPILKAN RANKING

# Grouping berdasarkan skor
grouped = {}
for idx, doc, score in results:
    grouped.setdefault(score, []).append((idx, doc))

# Tampilkan hasil
print(f"\n{'='*60}")
print(f"  HASIL PENCARIAN untuk query: '{query_input}'")
print(f"{'='*60}")

rank = 1
shown = 0

for score, docs in grouped.items():
    if score == 0:
        continue    # skip dokumen yang skornya 0 (tidak relevan)

    doc_ids = ', '.join([f"D{d[0]}" for d in docs])
    print(f"\nRank {rank} | Dokumen: {doc_ids} | Skor: {score}")
    print("-" * 60)

    for d in docs:
        # Tampilkan potongan teks (maks 120 karakter)
        preview = d[1][:120] + "..." if len(d[1]) > 120 else d[1]
        print(f"  D{d[0]}: {preview}")

    rank += 1
    shown += 1

if shown == 0:
    print("\nTidak ada dokumen yang relevan dengan query tersebut.")

print(f"\n{'='*60}")


  HASIL PENCARIAN untuk query: 'economy'

Rank 1 | Dokumen: D5, D11, D12, D13, D14, D15, D16, D17, D18, D19, D20, D21, D22, D23, D24, D25, D26, D27, D28, D29, D30, D32, D33, D34, D35, D37, D38, D39, D40, D41, D42, D43, D44, D45, D46, D47, D48, D49, D50 | Skor: 1.22215
------------------------------------------------------------
  D5: green economy to create new avenues of green jobs and green entrepreneurship youth to play key role in india sustainabil...
  D11: very pleased to catch up with public policy team to discuss changes in the uk financial services industry and the import...
  D12: no more talk nccc dg pushes urgent billions in climate finance to power nigeria green economy
  D13: from tissue culture to handicrafts bamboo innovation is transforming livelihoods across the northeast technology value a...
  D14: energy transition and the green economy cannot be the solution to this crisis the upward pressure on input costs for the...
  D15: sustainable growth green economy expec